# Lakehouse — Başlangıç Notebook'u

Bu pod, kimliğiniz (`DEPT`) ve lakehouse bağlantı bilgileriniz önceden enjekte edilmiş şekilde
açıldı — ek konfigürasyon gerekmez. Üç sorgu motoru da hazır, hepsi `lakehouse_nb` yardımcı
modülü üzerinden (`import lakehouse_nb as lh`):

- **PyIceberg** (`lh.iceberg()`) — REST katalog üzerinden doğrudan Python/pandas okuma.
- **Spark** (`lh.spark()`) — `lakehouse` (prod Silver) ve `rawlake` katalogları salt-okuma;
  kendi `sandbox_<DEPT>` kataloğunuza okuma/yazma.
- **Trino** (`lh.trino().cursor()`) — hızlı SQL, salt-okuma (`lakehouse` kataloğu).

Aşağıdaki tablo adları **placeholder**'dır (`<ns>.<tablo>`) — bu notebook ürün-generic'tir,
belirli bir demo şemasına bağlı değildir. Gerçek şema/tablo adlarınızı görmek için en alttaki
"kendi şema ve tablolarını listele" hücresini çalıştırın.

In [ ]:
import lakehouse_nb as lh
import os

print(os.environ.get("DEPT"))

## 1) PyIceberg — REST katalogdan doğrudan oku

`<ns>.<tablo>` yerine kendi namespace/tablo adınızı yazın (aşağıdaki listeleme hücresiyle
bulabilirsiniz).

In [ ]:
cat = lh.iceberg()
cat.list_namespaces()

In [ ]:
# Placeholder: <ns>.<tablo> -> kendi namespace/tablo adınızla değiştirin
cat.load_table("<ns>.<tablo>").scan().to_pandas().head()

## 2) Spark — prod Silver'ı oku, kendi sandbox'ına yaz

`lakehouse` ve `rawlake` katalogları salt-okuma prod verisidir. Kendi denemeleriniz için
`sandbox_<DEPT>` kataloğunuza (aşağıdaki yorumlu örnek) yazabilirsiniz.

In [ ]:
s = lh.spark()
s.sql("SHOW SCHEMAS IN lakehouse").show()

In [ ]:
# Placeholder: <ns>.<tablo> -> kendi namespace/tablo adınızla değiştirin.
# Yazım örneği (varsayılan olarak yorum satırı -- kendi sandbox kataloğunuza yazar):
#
# df = s.table("lakehouse.<ns>.<tablo>")
# sandbox_catalog = os.environ["NESSIE_SANDBOX_CATALOG"]  # örn. "sandbox_veri"
# df.writeTo(f"{sandbox_catalog}.<ns>.<tablo>").createOrReplace()

## 3) Trino — hızlı SQL (salt-okuma)

In [ ]:
# Placeholder: <ns>.<tablo> -> kendi namespace/tablo adınızla değiştirin
c = lh.trino().cursor()
c.execute("SELECT count(*) FROM lakehouse.<ns>.<tablo>")
c.fetchone()

## 4) Kendi şema/tablolarınızı keşfedin

Yukarıdaki hücrelerdeki `<ns>.<tablo>` placeholder'larını gerçek adlarla değiştirmeden önce,
hangi şema ve tabloların var olduğunu görmek için bu hücreyi çalıştırın (Spark veya Trino ile).

In [ ]:
# Spark ile:
s.sql("SHOW SCHEMAS IN lakehouse").show()
s.sql("SHOW TABLES IN lakehouse.<ns>").show()

# Trino ile (eşdeğer):
# c.execute("SHOW SCHEMAS FROM lakehouse"); c.fetchall()
# c.execute("SHOW TABLES FROM lakehouse.<ns>"); c.fetchall()